In [ ]:
#point the directory to your main directory for the project
PROTGPS_PARENT_DIR = "/home/shd-sun-lab/SynapseNavigator" # point to the protgps local repo

In [ ]:
#see if GPU is available for PyTorch
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA version (PyTorch built with):", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
CUDA version (PyTorch built with): 13.0
GPU: NVIDIA GeForce RTX 4070


In [ ]:
#import dependencies
import sys
import os
sys.path.append(PROTGPS_PARENT_DIR) # append the path of protgps
from argparse import Namespace
import pickle
from tqdm import tqdm
import pandas as pd
import torch 
from protgps.utils.loading import get_object
from tkinter import Tk, filedialog
from esmc_600m.models.esmc_encoder import ESMCEncoder
import matplotlib.pyplot as plt
import seaborn as sns

/home/shd-sun-lab/miniforge3/envs/syna_esmc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
#load the model
COMPARTMENT_CLASSES = [
            "cytosol",
            "ER",
            "mitochondrion",
            "nucleus",
            "Excitatory Synapse",
            "Inhibitory Synapses"
]

def load_model(snargs):
    """
    Loads classifier model from args file
    """
    modelpath = snargs.model_path
    model = get_object(snargs.lightning_name, "lightning")(snargs)
    model = model.load_from_checkpoint(
        checkpoint_path = modelpath,
        strict=not snargs.relax_checkpoint_matching,
        **{"args": snargs},
    )
    return model

@torch.no_grad()
def predict_condensates(model, sequences, batch_size=1, round=True):
    scores = []
    for i in tqdm(range(0, len(sequences), batch_size), ncols=100):
        batch = sequences[ i : (i + batch_size)]
        out = model.model({"x": batch})    
        s = torch.sigmoid(out['logit']).to("cpu")
        scores.append(s)
    scores = torch.vstack(scores)
    if round:
        scores = torch.round(scores, decimals=3)
    return scores

In [7]:
# Allow argparse.Namespace for safe unpickling (PyTorch 2.6+ requirement)
torch.serialization.add_safe_globals([Namespace])

# Use a file dialog to select the .args file
Tk().withdraw()  # Hide the main tkinter window
args_path = filedialog.askopenfilename(title="Select .args file", filetypes=[("Args files", "*.args")])

# Load args
args = Namespace(**pickle.load(open(args_path, 'rb')))

# Prompt to select the .ckpt file
ckpt_path = filedialog.askopenfilename(title="Select .ckpt file", filetypes=[("Checkpoint files", "*.ckpt")])
args.model_path = ckpt_path

# Set the pretrained hub directory manually (if static)
args.pretrained_hub_dir = "/home/shd-sun-lab/protgps/checkpoints/ESM-C/protgps_esmc_A2"

# Load and prepare model
model = load_model(args)
model.eval()
model = model.to(device)


/home/shd-sun-lab/miniforge3/envs/syna_esmc/lib/python3.10/site-packages/pytorch_lightning/core/lightning.py:2067: DeprecationWarning: `torch.distributed._sharded_tensor` will be deprecated, use `torch.distributed._shard.sharded_tensor` instead
  from torch.distributed._sharded_tensor import pre_load_state_dict_hook, state_dict_hook


Loading ESM-C model: esmc_600m
  Device: cuda
  Freeze encoder: False


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 59918.63it/s]


  Fine-tuning ESM-C encoder (all parameters trainable)
  Hidden size: 1152 (default)
  Representation layer: -1
ESM-C encoder initialized successfully!
Loading ESM-C model: esmc_600m
  Device: cuda
  Freeze encoder: False
  Fine-tuning ESM-C encoder (all parameters trainable)
  Hidden size: 1152 (default)
  Representation layer: -1
ESM-C encoder initialized successfully!


Down below is prediction of genes

In [ ]:
#optional code for when you have larger datasets of proteins you want to test, you can load them from an excel file with the following code. The excel file should have two columns: "Sequences" and "protein_names". The sequences will be used for prediction, and the protein names will be used for labeling the results.
# Load sequences and protein names from Excel
Tk().withdraw()
excel_path = filedialog.askopenfilename(title="fulL_E3_list", filetypes=[("Excel files", "*.xlsx *.xls")])

df_input = pd.read_excel(excel_path)
sequences = df_input["Sequences"].dropna().tolist()
protein_names = df_input["protein_names"].dropna().tolist()

ValueError: Invalid file path or buffer object type: <class 'tuple'>

In [8]:
#Predict sequences
sequences = [
    #Gene_name1
    "PPPXXXXXXAPAGPAXSXSXTG"
]


# Add protein names corresponding to the sequences
protein_names = [
    "seq1",
]

In [9]:
scores = predict_condensates(model, sequences, batch_size=1)

100%|█████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.42it/s]


In [10]:
data = {"Protein": protein_names, "sequences": sequences}
for j, condensate in enumerate(COMPARTMENT_CLASSES):
    data[f"{condensate.upper()}_Score"] = scores[:, j].tolist()

In [11]:
pd.DataFrame(data)

,Protein,sequences,CYTOSOL_Score,ER_Score,MITOCHONDRION_Score,NUCLEUS_Score,EXCITATORY SYNAPSE_Score,INHIBITORY SYNAPSES_Score
0,seq1,PPPXXXXXXAPAGPAXSXSXTG,0.457,0.08,0.037,0.911,0.352,0.028


In [12]:
df_output = pd.DataFrame(data)

output_path = filedialog.asksaveasfilename(
    title="motif_predictions",
    defaultextension=".xlsx",
    filetypes=[("Excel files", "*.xlsx")]
)

df_output.to_excel(output_path, index=False)
print(f"Saved to {output_path}")

Saved to /home/shd-sun-lab/SynapseNavigator/markus_output/SynGO_SynapsePredicted_Proteins.xlsx


Masking motif analysis

In [12]:
# Load dataset
import json

DATASET_PATH = "/home/shd-sun-lab/SynapseNavigator/data/dataset.json"
ESMC_COMPS = ["Cytosol", "ER", "Mitochondrion", "Nucleus", 
              "Excitatory Synapse", "Inhibitory Synapse"]

print(f"\nLoading dataset from: {DATASET_PATH}")
with open(DATASET_PATH, 'r') as f:
    dataset_full = json.load(f)


# Filter test set proteins
test_proteins = {p['Entry Name']: p for p in dataset_full if p.get('split') == 'test'}
print(f"✓ Test set: {len(test_proteins)} proteins")


Loading dataset from: /home/shd-sun-lab/SynapseNavigator/data/dataset.json
✓ Test set: 1015 proteins


In [13]:
OUTPUT_DIR = "/home/shd-sun-lab/SynapseNavigator/notebook/ESMC_outputs"

# Compartment names
COMPARTMENTS = [
    "Cytosol",
    "ER",
    "Mitochondrion",
    "Nucleus",
    "Excitatory Synapse",
    "Inhibitory Synapses"
]

In [14]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch

# ==========================================
# 1. DEFINE YOUR PROTEINS DIRECTLY
# ==========================================
TARGET_PROTEINS = {
    "Excitatory Synapse": {
        "NEDD4L": "MATGLGEPVYGLSEDEGESRILRVKVVSGIDLAKKDIFGASDPYVKLSLYVADENRELALVQTKTIKKTLNPKWNEEFYFRVNPSNHRLLFEVFDENRLTRDDFLGQVDVPLSHLPTEDPTMERPYTFKDFLLRPRSHKSRVKGFLRLKMAYMPKNGGQDEENSDQRDDMEHGWEVVDSNDSASQHQEELPPPPLPPGWEEKVDNLGRTYYVNHNNRTTQWHRPSLMDVSSESDNNIRQINQEAAHRRFRSRRHISEDLEPEPSEGGDVPEPWETISEEVNIAGDSLGLALPPPPASPGSRTSPQELSEELSRRLQITPDSNGEQFSSLIQREPSSRLRSCSVTDAVAEQGHLPPPSAPAGRARSSTVTGGEEPTPSVAYVHTTPGLPSGWEERKDAKGRTYYVNHNNRTTTWTRPIMQLAEDGASGSATNSNNHLIEPQIRRPRSLSSPTVTLSAPLEGAKDSPVRRAVKDTLSNPQSPQPSPYNSPKPQHKVTQSFLPPGWEMRIAPNGRPFFIDHNTKTTTWEDPRLKFPVHMRSKTSLNPNDLGPLPPGWEERIHLDGRTFYIDHNSKITQWEDPRLQNPAITGPAVPYSREFKQKYDYFRKKLKKPADIPNRFEMKLHRNNIFEESYRRIMSVKRPDVLKARLWIEFESEKGLDYGGVAREWFFLLSKEMFNPYYGLFEYSATDNYTLQINPNSGLCNEDHLSYFTFIGRVAGLAVFHGKLLDGFFIRPFYKMMLGKQITLNDMESVDSEYYNSLKWILENDPTELDLMFCIDEENFGQTYQVDLKPNGSEIMVTNENKREYIDLVIQWRFVNRVQKQMNAFLEGFTELLPIDLIKIFDENELELLMCGLGDVDVNDWRQHSIYKNGYCPNHPVIQWFWKAVLLMDAEKRIRLLQFVTGTSRVPMNGFAELYGSNGPQLFTIEQWGSPEKLPRAHTCFNRLDLPPYETFEDLREKLLMAVENAQGFEGVD",
        # add more proteins here
        # "SHANK3": "MAST...",
    }
}

In [15]:
# 2. MASKING-BASED RESIDUE IMPORTANCE
# ==========================================
def get_residue_importance_by_masking(model, sequence, target_class, device=device):
    """
    Calculate residue importance by masking each position and measuring score change

    Returns:
        importance_scores: Array of importance scores (higher = more important)
    """
    model.eval()

    # Get baseline prediction (full sequence)
    with torch.no_grad():
        batch = {"x": [sequence]}
        output = model.model(batch)
        baseline_score = torch.sigmoid(output["logit"][0, target_class]).item()

    importance_scores = []

    # Mask each residue and measure score drop
    for i in tqdm(range(len(sequence)), desc="Masking residues", leave=False):
        # Create masked sequence (replace with 'X')
        masked_seq = sequence[:i] + 'X' + sequence[i+1:]

        with torch.no_grad():
            batch = {"x": [masked_seq]}
            output = model.model(batch)
            masked_score = torch.sigmoid(output["logit"][0, target_class]).item()

        # Importance = drop in score when masked
        importance = baseline_score - masked_score
        importance_scores.append(importance)

    return np.array(importance_scores), baseline_score

# ==========================================
# 3. ANALYZE TARGET PROTEINS
# ==========================================
print("\n" + "="*80)
print("ANALYZING TARGET PROTEINS")
print("="*80)

all_results = []

for compartment, proteins in TARGET_PROTEINS.items():
    target_idx = ESMC_COMPS.index(compartment)
    print(f"\n{'='*80}")
    print(f"Compartment: {compartment} (index {target_idx})")
    print(f"{'='*80}")

    for protein_name, sequence in proteins.items():
        print(f"\n{protein_name}:")
        print(f"  Length: {len(sequence)} aa")

        # Full probability breakdown before masking
        with torch.no_grad():
            out = model.model({"x": [sequence]})
        probs = torch.sigmoid(out["logit"]).detach().cpu()
        print("  Compartment probabilities:")
        for comp, prob in zip(ESMC_COMPS, probs[0]):
            marker = " ← target" if comp == compartment else ""
            print(f"    {comp:25s}: {prob:.4f}{marker}")

        # Calculate residue importance
        importance, baseline_score = get_residue_importance_by_masking(
            model, sequence, target_idx, device=device
        )

        print(f"  Baseline score:  {baseline_score:.4f}")
        print(f"  Max importance:  {importance.max():.4f} at position {importance.argmax()}")
        print(f"  Mean importance: {importance.mean():.4f}")

        # Regional analysis
        seq_len = len(sequence)
        n_term  = importance[:50] if seq_len >= 50 else importance[:seq_len//4]
        c_term  = importance[-50:] if seq_len >= 50 else importance[-seq_len//4:]
        middle  = importance[seq_len//4:-seq_len//4] if seq_len >= 100 else importance

        print(f"  N-terminus (1-50):  mean={n_term.mean():.4f}, max={n_term.max():.4f}")
        print(f"  C-terminus (-50):   mean={c_term.mean():.4f}, max={c_term.max():.4f}")
        print(f"  Middle:             mean={middle.mean():.4f}, max={middle.max():.4f}")

        # Save per-residue results
        result_df = pd.DataFrame({
            'Position':       np.arange(1, seq_len + 1),
            'Residue':        list(sequence),
            'Importance':     importance,
            'Baseline_Score': baseline_score
        })
        print(result_df[result_df['Importance'] > importance.mean() + importance.std()])

        all_results.append({
            'Protein':        protein_name,
            'Compartment':    compartment,
            'Length':         seq_len,
            'Baseline_Score': baseline_score,
            'Max_Importance': importance.max(),
            'Mean_Importance':importance.mean(),
            'N_term_Mean':    n_term.mean(),
            'C_term_Mean':    c_term.mean(),
            'Middle_Mean':    middle.mean(),
            'C_to_N_Ratio':   c_term.mean() / n_term.mean() if n_term.mean() > 0 else 0
        })

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
pd.DataFrame(all_results)


ANALYZING TARGET PROTEINS

Compartment: Excitatory Synapse (index 4)

NEDD4L:
  Length: 975 aa
  Compartment probabilities:
    Cytosol                  : 0.8486
    ER                       : 0.0768
    Mitochondrion            : 0.0022
    Nucleus                  : 0.1155
    Excitatory Synapse       : 0.6626 ← target
    Inhibitory Synapse       : 0.7657


  Baseline score:  0.6626
  Max importance:  0.1483 at position 391
  Mean importance: 0.0963
  N-terminus (1-50):  mean=0.0949, max=0.1407
  C-terminus (-50):   mean=0.1235, max=0.1470
  Middle:             mean=0.0920, max=0.1483
     Position Residue  Importance  Baseline_Score
2           3       T    0.119871        0.662584
3           4       G    0.140692        0.662584
5           6       G    0.123408        0.662584
7           8       P    0.136328        0.662584
10         11       G    0.133965        0.662584
..        ...     ...         ...             ...
964       965       V    0.120929        0.662584
965       966       E    0.126864        0.662584
966       967       N    0.124351        0.662584
967       968       A    0.121739        0.662584
968       969       Q    0.126893        0.662584

[139 rows x 4 columns]

SUMMARY


,Protein,Compartment,Length,Baseline_Score,Max_Importance,Mean_Importance,N_term_Mean,C_term_Mean,Middle_Mean,C_to_N_Ratio
0,NEDD4L,Excitatory Synapse,975,0.662584,0.148346,0.09626,0.094902,0.123539,0.091958,1.301757


In [16]:
# ==========================================
# 4. CREATE VISUALIZATION (Captum-style)
# ==========================================
from captum.attr import visualization as viz

records = []

# Z-score normalization for better visual contrast
z = (importance - importance.mean()) / (importance.std() + 1e-8)
z = np.clip(z, -3, 3)

record = viz.VisualizationDataRecord(
    word_attributions=z,
    pred_prob=float(baseline_score),
    pred_class=compartment,
    true_class=protein_name,
    attr_class="-",
    attr_score=float(importance.sum()),
    raw_input_ids=list(sequence),
    convergence_score=0.0,
)
records.append(record)

print("\n--- Captum Visualization ---")
viz.visualize_text(records)

# Define file paths
csv_file = f"{OUTPUT_DIR}/{compartment.replace(' ', '_')}_{protein_name}.csv"

# Save CSV
result_df = pd.DataFrame({
    'Position':       np.arange(1, len(sequence) + 1),
    'Residue':        list(sequence),
    'Importance':     importance,
    'Baseline_Score': baseline_score
})
result_df.to_csv(csv_file, index=False)
print(f"  ✓ Saved: {csv_file}")

# ==========================================
# 5. SUMMARY ANALYSIS
# ==========================================
print("\n" + "="*80)
print("SUMMARY: REGIONAL IMPORTANCE ANALYSIS")
print("="*80)

summary_df = pd.DataFrame(all_results)

for compartment in ["Inhibitory Synapses", "Excitatory Synapse"]:
    comp_data = summary_df[summary_df['Compartment'] == compartment]
    if len(comp_data) == 0:
        continue
    print(f"\n{compartment}:")
    print(f"  Average C-terminus importance: {comp_data['C_term_Mean'].mean():.4f}")
    print(f"  Average N-terminus importance: {comp_data['N_term_Mean'].mean():.4f}")
    print(f"  Average C-to-N ratio: {comp_data['C_to_N_Ratio'].mean():.2f}x")
    print(f"  Proteins with C>N (ratio>1.5):")
    for _, row in comp_data.iterrows():
        if row['C_to_N_Ratio'] > 1.5:
            print(f"    • {row['Protein']}: {row['C_to_N_Ratio']:.2f}x (C={row['C_term_Mean']:.4f}, N={row['N_term_Mean']:.4f})")

# Save summary
summary_file = f"{OUTPUT_DIR}/motif_analysis_summary.csv"
summary_df.to_csv(summary_file, index=False)
print(f"\n✓ Summary saved: {summary_file}")

print("\n" + "="*80)
print("MOTIF ANALYSIS COMPLETE!")
print("="*80)
print(f"Results saved to: {OUTPUT_DIR}/")


--- Captum Visualization ---


  ✓ Saved: /home/shd-sun-lab/SynapseNavigator/notebook/ESMC_outputs/Excitatory_Synapse_NEDD4L.csv

SUMMARY: REGIONAL IMPORTANCE ANALYSIS

Excitatory Synapse:
  Average C-terminus importance: 0.1235
  Average N-terminus importance: 0.0949
  Average C-to-N ratio: 1.30x
  Proteins with C>N (ratio>1.5):

✓ Summary saved: /home/shd-sun-lab/SynapseNavigator/notebook/ESMC_outputs/motif_analysis_summary.csv

MOTIF ANALYSIS COMPLETE!
Results saved to: /home/shd-sun-lab/SynapseNavigator/notebook/ESMC_outputs/
